In [13]:
!pip install mlxtend

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 22.4 MB/s  0:00:00 24.1 MB/s eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.1
    Uninstalling numpy-1.26.1:
      Successfully uninstalled numpy-1.26.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mlxtend]


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
from sklearn.datasets import load_breast_cancer

data_clf = load_breast_cancer(as_frame=True)
df = data_clf.frame

In [3]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
x_clf = df.drop('target', axis = 1)
y_clf = df['target']

In [5]:
x_train_clf, x_test_clf, y_train_clf, y_test_clf = train_test_split(x_clf, y_clf, test_size = 0.2, random_state = 42)

In [6]:
sc = StandardScaler()

x_train_clf = sc.fit_transform(x_train_clf)
x_test_clf = sc.transform(x_test_clf)

# Constructing Base Models

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

KNN = KNeighborsClassifier()   #Uses 5 nearest neighbours
DT = DecisionTreeClassifier(max_depth=3, random_state=42) #Shallow Tree

In [8]:
#Train KNN
KNN.fit(x_train_clf, y_train_clf)
knn_pred = KNN.predict(x_test_clf)
knn_acc_clf = accuracy_score(y_test_clf, knn_pred)

#Train Decision Tree
DT.fit(x_train_clf, y_train_clf)
dt_pred = DT.predict(x_test_clf)
dt_acc_clf = accuracy_score(y_test_clf, dt_pred)

print('Accuracy Score of KNeighbors Classifier:', knn_acc_clf)
print('Accuracy of Decision Tree Classifier:', dt_acc_clf)

Accuracy Score of KNeighbors Classifier: 0.9473684210526315
Accuracy of Decision Tree Classifier: 0.9473684210526315


# Implementing the Stacking Classifier

In [14]:
# Build Stacking Classifier
from mlxtend.classifier import StackingClassifier
from sklearn.linear_model import LogisticRegression

base_learners = [
    KNeighborsClassifier(),
    DecisionTreeClassifier(max_depth=3, random_state=42)
]

meta_model = LogisticRegression() #logistic regression as meta model

stacking_model = StackingClassifier(classifiers=base_learners, meta_classifier=meta_model, use_probas=True)

In [15]:
#Train Stacking Classifier
stacking_model.fit(x_train_clf, y_train_clf)
pred_stack = stacking_model.predict(x_test_clf)

acc_stack = accuracy_score(y_test_clf, pred_stack)
print('Accuracy Score of Stacked Model:', acc_stack)

Accuracy Score of Stacked Model: 0.956140350877193


In [16]:
results = pd.DataFrame({
    'Model': [
        'K-Nearest Neighbors',
        'Decision Tree',
        'Stacking Classifier'
    ],
    'Accuracy': [
        knn_acc_clf,
        dt_acc_clf,
        acc_stack
    ]
})

results

,Model,Accuracy
0,K-Nearest Neighbors,0.947368
1,Decision Tree,0.947368
2,Stacking Classifier,0.956140


# Conclusion
- We trained K-Nearest Neighbors and Decision Tree models on the breast cancer dataset and evaluated their individual accuracies. We then applied a stacking classifier using logistic regression as the meta-model and compared its accuracy with the base models to observe how stacking combines their strengths and affects performance.

# Regression
- We are given the California housing dataset containing various numerical features related to housing characteristics. Using this data, we aim to build machine learning models to predict house prices. First, we train individual regression models, and then we construct a stacking regressor that combines these models to improve prediction performance.

In [19]:
from sklearn.datasets import fetch_california_housing

data_reg = fetch_california_housing(as_frame=True)

x_reg = data_reg.data
y_reg = data_reg.target

In [20]:
x_reg.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [21]:
y_reg.head()

0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64

In [22]:
x_train_reg, x_test_reg, y_train_reg, y_test_reg = train_test_split(x_reg, y_reg, test_size=0.2, random_state=42)

In [23]:
scaler = StandardScaler()

x_train_reg = scaler.fit_transform(x_train_reg)
x_test_reg = scaler.transform(x_test_reg)

# Constructing base models

In [24]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

knn_reg = KNeighborsRegressor(n_neighbors=5)   #Uses 5 nearest neighbours
lr = LinearRegression()   #Linear Baseline model
dtr = DecisionTreeRegressor(max_depth=8, random_state=42)

In [25]:
from sklearn.metrics import mean_squared_error

#train KNN
knn_reg.fit(x_train_reg, y_train_reg)
knn_reg_pred = knn_reg.predict(x_test_reg)
knn_rmse = np.sqrt(mean_squared_error(y_test_reg, knn_reg_pred))

#Train Logistic Regresson
lr.fit(x_train_reg, y_train_reg)
lr_pred = lr.predict(x_test_reg)
lr_rmse = np.sqrt(mean_squared_error(y_test_reg, lr_pred))

#Train Decision Tree Regressor
dtr.fit(x_train_reg, y_train_reg)
dtr_pred = dtr.predict(x_test_reg)
dt_rmse = np.sqrt(mean_squared_error(y_test_reg, dtr_pred))

print("KNN RMSE:", knn_rmse)
print("Linear Regression RMSE:", lr_rmse)
print("Decision Tree RMSE:", dt_rmse)

KNN RMSE: 0.6575877238850522
Linear Regression RMSE: 0.7455813830127764
Decision Tree RMSE: 0.6496502038503689


In [30]:
# Generate base model predictions
knn_meta_pred = knn_reg.predict(x_test_reg)
lr_meta_pred = lr.predict(x_test_reg)
dt_meta_pred = dtr.predict(x_test_reg)

# Stack predictions
meta_features_reg = np.column_stack([
    knn_meta_pred,
    lr_meta_pred,
    dt_meta_pred
])

# Convert to DataFrame
meta_features_reg_df = pd.DataFrame(
    meta_features_reg,
    columns=[
        'KNN_Prediction',
        'LinearRegression_Prediction',
        'DecisionTree_Prediction'
    ]
)

print("Meta-model input features (first 5 samples):")
meta_features_reg_df.head()

Meta-model input features (first 5 samples):


,KNN_Prediction,LinearRegression_Prediction,DecisionTree_Prediction
0,0.498800,0.719123,0.688495
1,0.764600,1.764017,0.804917
2,4.750006,2.709659,4.107126
3,2.876600,2.838926,2.430196
4,2.726200,2.604657,1.522334


In [31]:
meta_features_reg

array([[0.4988    , 0.71912284, 0.68849463],
       [0.7646    , 1.76401657, 0.80491743],
       [4.750006  , 2.70965883, 4.10712575],
       ...,
       [4.761208  , 4.46877017, 4.96077044],
       [0.692     , 1.18751119, 0.80491743],
       [1.8944    , 2.00940251, 1.63388889]])

In [26]:
from mlxtend.regressor import StackingRegressor
from sklearn.linear_model import Ridge

base_regressors = [
    KNeighborsRegressor(n_neighbors=5),
    DecisionTreeRegressor(max_depth=8, random_state=42),
    LinearRegression()
]

meta_regressor = Ridge()

stacking_reg = StackingRegressor(
    regressors=base_regressors,
    meta_regressor=meta_regressor
)

Why the Stacking regressor takes the x_train_reg instead of meta_features_reg?
- the reason is that StackingRegressor in sklearn automatically creates the meta-features internally. You don't need to manually pass meta_features_reg.

In [27]:
#Train Stacing Regressor
stacking_reg.fit(x_train_reg, y_train_reg)
stack_pred = stacking_reg.predict(x_test_reg)

In [28]:
stacking_rmse = np.sqrt(mean_squared_error(y_test_reg, stack_pred))

print("Stacked Model RMSE:", stacking_rmse)

Stacked Model RMSE: 0.6072082936910247


In [29]:
results = pd.DataFrame({
    'Model': [
        'K-Nearest Neighbors',
        'Linear Regression',
        'Decision Tree',
        'Stacking Regressor'
    ],
    'RMSE': [
        knn_rmse,
        lr_rmse,
        dt_rmse,
        stacking_rmse
    ]
})

results

,Model,RMSE
0,K-Nearest Neighbors,0.657588
1,Linear Regression,0.745581
2,Decision Tree,0.649650
3,Stacking Regressor,0.607208


# Conclusion
- We trained K-Nearest Neighbors, Linear Regression, and Decision Tree models on the California housing dataset and evaluated their performance using RMSE. We then applied a stacking regressor that combines these base models using a Ridge meta-model. By comparing the RMSE values, we observed how stacking leverages the strengths of individual models to achieve improved prediction performance.